In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os
os.chdir("/content")
print(os.getcwd())


/content


In [3]:
!rm -rf /content/Task-driven-low-light-enhancement
!git clone https://github.com/SarthakBaghel/Task-driven-low-light-enhancement.git /content/Task-driven-low-light-enhancement
%cd /content/Task-driven-low-light-enhancement
!pip install -r requirements.txt


Cloning into '/content/Task-driven-low-light-enhancement'...
remote: Enumerating objects: 72, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 72 (delta 15), reused 67 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (72/72), 158.34 KiB | 2.93 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/Task-driven-low-light-enhancement
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 8.9 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [4]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


CUDA available: False


In [5]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
PIPELINE_ROOT = DRIVE_ROOT / "task_driven_video_pipeline"
KAGGLE_V1_ROOT = PIPELINE_ROOT / "kaggle_v1"
CHECKPOINTS_ROOT = DRIVE_ROOT / "task_driven_checkpoints" / "kaggle_v1"

RUN_TAG = "subject_class_balanced_20k"

TEST_CLEAN_ROOT = KAGGLE_V1_ROOT / f"test_clean_{RUN_TAG}"
TEST_LOWLIGHT_ROOT = KAGGLE_V1_ROOT / f"test_lowlight_{RUN_TAG}_eye_mid"

FULL_CKPT_DIR = CHECKPOINTS_ROOT / f"full_clean_detector_{RUN_TAG}_resnet18"
FULL_BEST_CKPT = FULL_CKPT_DIR / f"kaggle_v1_clean_full_{RUN_TAG}_best.pt"

BASELINE_REPORT_DIR = KAGGLE_V1_ROOT / f"eval_clean_vs_lowlight_test_{RUN_TAG}_eye_mid_5k"

assert TEST_CLEAN_ROOT.exists(), f"Missing: {TEST_CLEAN_ROOT}"
assert TEST_LOWLIGHT_ROOT.exists(), f"Missing: {TEST_LOWLIGHT_ROOT}"
assert FULL_BEST_CKPT.exists(), f"Missing: {FULL_BEST_CKPT}"

print("TEST_CLEAN_ROOT:", TEST_CLEAN_ROOT)
print("TEST_LOWLIGHT_ROOT:", TEST_LOWLIGHT_ROOT)
print("FULL_BEST_CKPT:", FULL_BEST_CKPT)
print("BASELINE_REPORT_DIR:", BASELINE_REPORT_DIR)


TEST_CLEAN_ROOT: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/test_clean_subject_class_balanced_20k
TEST_LOWLIGHT_ROOT: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/test_lowlight_subject_class_balanced_20k_eye_mid
FULL_BEST_CKPT: /content/drive/MyDrive/task_driven_checkpoints/kaggle_v1/full_clean_detector_subject_class_balanced_20k_resnet18/kaggle_v1_clean_full_subject_class_balanced_20k_best.pt
BASELINE_REPORT_DIR: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_clean_vs_lowlight_test_subject_class_balanced_20k_eye_mid_5k


In [6]:
!find "{TEST_CLEAN_ROOT}/open" -type f | wc -l
!find "{TEST_CLEAN_ROOT}/closed" -type f | wc -l

!find "{TEST_LOWLIGHT_ROOT}/open" -type f | wc -l
!find "{TEST_LOWLIGHT_ROOT}/closed" -type f | wc -l


8644
8644
8644
8644


In [7]:
!python3 /content/Task-driven-low-light-enhancement/evaluate_transfer_detector.py \
  {FULL_BEST_CKPT} \
  {TEST_CLEAN_ROOT} \
  --lowlight-root {TEST_LOWLIGHT_ROOT} \
  --batch-size 64 \
  --num-workers 0 \
  --max-total-samples 5000 \
  --subset-seed 42 \
  --output-dir {BASELINE_REPORT_DIR}


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /content/Task-driven-low-light-enhancement/artifacts/torch_cache/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 222MB/s]
Clean: 100% 79/79 [21:53<00:00, 16.63s/batch]
Evaluating on device: cpu
Classes: {'open': 0, 'closed': 1}
Using balanced subset with 5000 samples total: {'open': 2500, 'closed': 2500}
Clean    loss: 0.1214 | accuracy: 0.9256 | precision: 0.8741 | recall: 0.9944 | f1: 0.9304 | closed_recall: 0.9944 | threshold: 0.70
Saved subset manifest to: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_clean_vs_lowlight_test_subject_class_balanced_20k_eye_mid_5k/subset_manifest.csv
LowLight: 100% 79/79 [30:56<00:00, 23.50s/batch]
LowLight loss: 0.0410 | accuracy: 0.8960 | precision: 0.9843 | recall: 0.8048 | f1: 0.8856 | closed_recall: 0.8048 | threshold: 0.70
Saved evaluation report to: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_clean_vs_lowlight_tes

In [8]:
!find "{BASELINE_REPORT_DIR}" -maxdepth 1 -type f | sed -n '1,50p'


/content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_clean_vs_lowlight_test_subject_class_balanced_20k_eye_mid_5k/subset_manifest.csv
/content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_clean_vs_lowlight_test_subject_class_balanced_20k_eye_mid_5k/clean_only_experiment_results.csv
/content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_clean_vs_lowlight_test_subject_class_balanced_20k_eye_mid_5k/clean_only_evaluation_results.csv
/content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_clean_vs_lowlight_test_subject_class_balanced_20k_eye_mid_5k/clean_only_evaluation_summary.txt
/content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_clean_vs_lowlight_test_subject_class_balanced_20k_eye_mid_5k/experiment_results.csv
/content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_clean_vs_lowlight_test_subject_class_balanced_20k_eye_mid_5k/evaluation_results.csv
/content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_clean_vs_lo

In [9]:
!sed -n '1,200p' "{BASELINE_REPORT_DIR}/evaluation_summary.txt"


Using threshold=0.70 for the closed-eye class.
Closed-eye recall on clean data: 0.9944.
Balanced subset used for evaluation: 5000 samples total.
Per-class subset counts: {'open': 2500, 'closed': 2500}.
Closed-eye recall on low-light data: 0.8048.
Low-light degradation summary: accuracy 92.56% -> 89.60%.
Accuracy: -2.96 points (drop).
Precision: +11.02 points (change).
Recall: -18.96 points (drop).
F1: -4.48 points (drop).

In [10]:
from pathlib import Path
import pandas as pd

csv_path = BASELINE_REPORT_DIR / "experiment_results.csv"
detailed_csv_path = BASELINE_REPORT_DIR / "evaluation_results.csv"
subset_manifest_path = BASELINE_REPORT_DIR / "subset_manifest.csv"
clean_only_csv_path = BASELINE_REPORT_DIR / "clean_only_experiment_results.csv"
clean_only_summary_path = BASELINE_REPORT_DIR / "clean_only_evaluation_summary.txt"

print("Compact CSV:", csv_path)
print("Detailed CSV:", detailed_csv_path)
print("Subset manifest:", subset_manifest_path)
print("Clean-only CSV:", clean_only_csv_path)
print("Clean-only summary:", clean_only_summary_path)

if clean_only_summary_path.exists():
    print("\n=== clean_only_evaluation_summary.txt ===")
    print(clean_only_summary_path.read_text())

if clean_only_csv_path.exists():
    print("\n=== clean_only_experiment_results.csv ===")
    df_clean = pd.read_csv(clean_only_csv_path)
    print(df_clean.to_string(index=False))

if csv_path.exists():
    print("\n=== experiment_results.csv ===")
    df = pd.read_csv(csv_path)
    print(df.to_string(index=False))

if detailed_csv_path.exists():
    print("\n=== evaluation_results.csv ===")
    detailed_df = pd.read_csv(detailed_csv_path)
    print(detailed_df.to_string(index=False))


Compact CSV: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_clean_vs_lowlight_test_subject_class_balanced_20k_eye_mid_5k/experiment_results.csv
Detailed CSV: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_clean_vs_lowlight_test_subject_class_balanced_20k_eye_mid_5k/evaluation_results.csv
Subset manifest: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_clean_vs_lowlight_test_subject_class_balanced_20k_eye_mid_5k/subset_manifest.csv
Clean-only CSV: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_clean_vs_lowlight_test_subject_class_balanced_20k_eye_mid_5k/clean_only_experiment_results.csv
Clean-only summary: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_clean_vs_lowlight_test_subject_class_balanced_20k_eye_mid_5k/clean_only_evaluation_summary.txt

=== clean_only_evaluation_summary.txt ===
Using threshold=0.70 for the closed-eye class.
Closed-eye recall on clean data: 0.9944.
Balanced subset used for 

In [10]:
from pathlib import Path
import pandas as pd

subset_manifest_path = BASELINE_REPORT_DIR / "subset_manifest.csv"

if subset_manifest_path.exists():
    subset_df = pd.read_csv(subset_manifest_path)
    print("Subset rows:", len(subset_df))
    display(subset_df.head(20))
    display(
        subset_df.groupby("class_name")
        .size()
        .reset_index(name="count")
        .sort_values("class_name")
    )
else:
    print("subset_manifest.csv not found")


In [11]:
print("Phase 6 complete.")
print("FULL_BEST_CKPT:", FULL_BEST_CKPT)
print("TEST_CLEAN_ROOT:", TEST_CLEAN_ROOT)
print("TEST_LOWLIGHT_ROOT:", TEST_LOWLIGHT_ROOT)
print("BASELINE_REPORT_DIR:", BASELINE_REPORT_DIR)


Phase 6 complete.
FULL_BEST_CKPT: /content/drive/MyDrive/task_driven_checkpoints/kaggle_v1/full_clean_detector_subject_class_balanced_20k_resnet18/kaggle_v1_clean_full_subject_class_balanced_20k_best.pt
TEST_CLEAN_ROOT: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/test_clean_subject_class_balanced_20k
TEST_LOWLIGHT_ROOT: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/test_lowlight_subject_class_balanced_20k_eye_mid
BASELINE_REPORT_DIR: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_clean_vs_lowlight_test_subject_class_balanced_20k_eye_mid_5k
